# 06 - Construção da tabela canônica

Constrói a tabela da inferência causal: **uma linha por par
`(within-paper group, estratégia ≠ none)`**, com o desfecho ΔY e as covariáveis Z.

- **Unidade:** par (grupo within-paper, estratégia ≠ none).
- **Grupo:** `within_paper_group_id` = (paper, dataset, modelo) - controla o "paper".
- **Desfecho:** ΔY relativo = `(Y_estratégia − Y_none) / Y_none`, com a mesma métrica μ_g no grupo.
- **Tratamento (4 níveis):** `oversampling`, `undersampling`, `cost_sensitive`, `other`
  (`none` é o baseline, consumido no ΔY). Principais: oversampling e cost_sensitive;
  exploratórios: undersampling e other.
- **Z:** IR, nº de classes, tamanho, tarefa, família de modelo - em versão **contínua**
  (para o ajuste do ATE) e **em faixas** (para o CATE), com flags de ausência (`*_missing`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

NB_DIR        = Path().resolve()
PROJECT_DIR   = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

INPUT_PATH  = PROCESSED_DIR / "experiments.parquet"
PAPERS_PATH = PROCESSED_DIR / "papers_with_pdf.parquet"
OUT_PATH    = PROCESSED_DIR / "analysis_dataset.parquet"

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 1. Carregar experimentos e restringir a grupos com baseline

Só grupos que contêm uma linha `none` (baseline) podem gerar ΔY.

In [2]:
df = pd.read_parquet(INPUT_PATH)
print(f"Experimentos:            {len(df):,}")
print(f"Grupos within-paper:     {df['within_paper_group_id'].nunique():,}")

df = df[df["has_baseline_in_group"]].copy()
print(f"Em grupos com baseline:  {len(df):,}")
print(f"Grupos com baseline:     {df['within_paper_group_id'].nunique():,}")

Experimentos:            1,309
Grupos within-paper:     676
Em grupos com baseline:  865
Grupos com baseline:     365


## 2. Normalizar a escala das métricas

Algumas métricas foram reportadas em escala 0–100. Corrige (÷100) onde o valor
estoura o intervalo [0,1] (limiar 1.5 para não tocar em valores legítimos).

In [3]:
METRICS_UNIT = [
    "metric_f1_macro", "metric_f1_binary", "metric_f1_micro", "metric_f1_weighted",
    "metric_balanced_acc", "metric_accuracy", "metric_aucroc", "metric_auprc",
    "metric_gmean",
]
fixed = 0
for m in METRICS_UNIT:
    if m not in df.columns:
        continue
    mask = df[m] > 1.5
    if mask.any():
        df.loc[mask, m] = df.loc[mask, m] / 100
        fixed += int(mask.sum())
print(f"Valores reescalados (÷100): {fixed}")

Valores reescalados (÷100): 16


## 3. Agregar estratégias em 4 níveis de tratamento

`none` = baseline. `oversampling` / `undersampling` / `cost_sensitive` permanecem;
todo o resto (threshold_moving, hybrid, generative, data_augmentation,
ensemble_based, two_stage) colapsa em `other`.

In [4]:
MAIN_STRATEGIES = {"oversampling", "undersampling", "cost_sensitive"}

def to_treatment(s):
    if s == "none":
        return "none"
    return s if s in MAIN_STRATEGIES else "other"

df["strategy"] = df["balancing_strategy"].map(to_treatment)
print(df["strategy"].value_counts().to_string())

strategy
none              374
other             204
oversampling      167
cost_sensitive     85
undersampling      35


## 4. Escolher uma métrica compartilhada por grupo (μ_g)

Dentro de cada grupo escolhe-se UMA métrica reportada tanto no baseline quanto em
≥1 tratamento, por ordem de prioridade (prefere métricas sensíveis a desbalanceamento,
com fallback para as mais comuns).

In [5]:
METRIC_PRIORITY = [
    "metric_f1_macro", "metric_balanced_acc", "metric_f1_binary", "metric_gmean",
    "metric_mcc", "metric_auprc", "metric_aucroc", "metric_accuracy",
    "metric_f1_micro", "metric_f1_weighted",
]

def shared_metric(group):
    base  = group[group["is_baseline"]]
    treat = group[~group["is_baseline"]]
    if base.empty or treat.empty:
        return None
    for m in METRIC_PRIORITY:
        if base[m].notna().any() and treat[m].notna().any():
            return m
    return None

## 5. Calcular o desfecho ΔY

Para cada `(grupo, estratégia ≠ none)`: agrega variantes (ex.: vários SMOTE) pela
**média** de μ_g e calcula ΔY relativo ao baseline. Descarta grupos sem métrica
compartilhada ou com baseline ≤ 0 (ΔY indefinido).

In [6]:
META_COLS = [
    "paper_id", "dataset_canonical", "model_family", "task_type",
    "dataset_imbalance_ratio", "dataset_num_classes", "dataset_size",
]

def first_valid(series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan

rows = []
skip_no_metric = skip_bad_baseline = 0
for gid, g in df.groupby("within_paper_group_id", sort=False):
    m = shared_metric(g)
    if m is None:
        skip_no_metric += 1
        continue
    y_base = g.loc[g["is_baseline"], m].mean()
    if pd.isna(y_base) or y_base <= 0:
        skip_bad_baseline += 1
        continue
    meta  = {c: first_valid(g[c]) for c in META_COLS}
    treat = g[(~g["is_baseline"]) & (g["strategy"] != "none")]
    for strat, sub in treat.groupby("strategy"):
        y_treat = sub[m].mean()          # agrega variantes pela média
        if pd.isna(y_treat):
            continue
        rows.append({
            "within_paper_group_id": int(gid),
            "strategy":    strat,
            "metric_used": m,
            "y_baseline":  float(y_base),
            "y_treatment": float(y_treat),
            "delta_y":     float((y_treat - y_base) / y_base),
            **meta,
        })

analysis = pd.DataFrame(rows)
print(f"Pares (grupo, estratégia):     {len(analysis):,}")
print(f"Grupos cobertos:               {analysis['within_paper_group_id'].nunique():,}")
print(f"Descartados sem métrica comum: {skip_no_metric:,}")
print(f"Descartados baseline <= 0:     {skip_bad_baseline:,}")
print()
print(analysis["strategy"].value_counts().to_string())
print()
print("ΔY:", {k: round(v, 3) for k, v in analysis["delta_y"].describe().to_dict().items()})

Pares (grupo, estratégia):     425
Grupos cobertos:               279
Descartados sem métrica comum: 70
Descartados baseline <= 0:     15

strategy
other             158
oversampling      156
cost_sensitive     79
undersampling      32

ΔY: {'count': 425.0, 'mean': 0.572, 'std': 4.354, 'min': -0.913, '25%': 0.002, '50%': 0.037, '75%': 0.131, 'max': 72.929}


## 6. Covariáveis Z: versão contínua + faixas + flags

Cada dimensão de Z aparece em duas formas: **contínua** (ajuste backdoor do ATE) e
**em faixas** (estratificação do CATE). Flags `*_missing` marcam ausência.

In [7]:
a = analysis

# --- IR ---
a["ir"]         = a["dataset_imbalance_ratio"]
a["ir_missing"] = a["ir"].isna()
a["ir_log"]     = np.log10(a["ir"].where(a["ir"] > 0))
IR_EDGES  = [1, 2, 5, 10, np.inf]
IR_LABELS = ["1-2", "2-5", "5-10", ">=10"]
a["ir_bin"] = pd.cut(a["ir"], bins=IR_EDGES, labels=IR_LABELS,
                     right=False, include_lowest=True).astype("object")
a.loc[a["ir_missing"], "ir_bin"] = "missing"

# --- nº de classes ---
a["n_classes"] = a["dataset_num_classes"]
a["is_binary"] = (a["n_classes"] == 2)

def bin_nclasses(n):
    if pd.isna(n): return "missing"
    if n <= 2:     return "2"
    if n <= 10:    return "3-10"
    return ">10"

a["nclasses_bin"] = a["n_classes"].map(bin_nclasses)

# --- tamanho (tercis log) ---
a["size_missing"] = a["dataset_size"].isna() | (a["dataset_size"] <= 0)
a["size_log"]     = np.log10(a["dataset_size"].where(~a["size_missing"]))
q1, q2 = a.loc[~a["size_missing"], "size_log"].quantile([1/3, 2/3])

def bin_size(row):
    if row["size_missing"]:      return "missing"
    if row["size_log"] < q1:     return "small"
    if row["size_log"] < q2:     return "medium"
    return "large"

a["size_bin"] = a.apply(bin_size, axis=1)

# --- tarefa -> 3 grupos (texto/time_series -> other) ---
TASK3 = {"tabular": "tabular", "vision": "vision"}
a["task3"] = a["task_type"].map(lambda t: TASK3.get(t, "other"))

# --- família de modelo -> 3 grupos (+ other) ---
MODEL_FAMILY3 = {
    "tree": "tree_ensemble", "ensemble": "tree_ensemble", "gbm": "tree_ensemble",
    "kernel": "kernel_linear", "linear": "kernel_linear",
    "cnn": "neural", "mlp": "neural", "rnn": "neural",
    "gnn": "neural", "transformer": "neural",
}
a["model_family3"] = a["model_family"].map(lambda m: MODEL_FAMILY3.get(m, "other"))

for col in ["ir_bin", "nclasses_bin", "size_bin", "task3", "model_family3"]:
    print(f"\n{col}:")
    print(a[col].value_counts(dropna=False).to_string())


ir_bin:
ir_bin
>=10       206
missing     76
1-2         59
2-5         55
5-10        29

nclasses_bin:
nclasses_bin
2          227
3-10       100
>10         83
missing     15

size_bin:
size_bin
missing    171
medium      85
large       85
small       84

task3:
task3
tabular    265
other      109
vision      51

model_family3:
model_family3
neural           224
tree_ensemble    109
kernel_linear     65
other             27


## 7. Anexar DOI e link do artigo, depois salvar

Mantém apenas as colunas usadas na análise, mais DOI e link do PDF (para voltar ao
artigo original quando necessário).

In [8]:
papers = (pd.read_parquet(PAPERS_PATH)[["paper_id", "doi", "oa_pdf_url"]]
          .drop_duplicates("paper_id")
          .rename(columns={"oa_pdf_url": "article_url"}))
a = a.merge(papers, on="paper_id", how="left")

OUT_COLS = [
    "within_paper_group_id", "paper_id", "doi", "article_url",
    "dataset_canonical", "strategy", "metric_used",
    "y_baseline", "y_treatment", "delta_y",
    "ir", "ir_log", "ir_bin", "ir_missing",
    "n_classes", "is_binary", "nclasses_bin",
    "dataset_size", "size_log", "size_bin", "size_missing",
    "task3", "model_family3",
]
final = a[OUT_COLS].copy()
final.to_parquet(OUT_PATH, index=False)
print(f"Salvo: {OUT_PATH}")
print(f"Linhas: {len(final):,}  |  colunas: {len(OUT_COLS)}")

Salvo: C:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\causal_project\data\processed\analysis_dataset.parquet
Linhas: 425  |  colunas: 23


## 8. Conferência do schema final

In [9]:
print(final.dtypes.to_string())
print()
final.head(8)

within_paper_group_id      int64
paper_id                  object
doi                       object
article_url               object
dataset_canonical         object
strategy                  object
metric_used               object
y_baseline               float64
y_treatment              float64
delta_y                  float64
ir                       float64
ir_log                   float64
ir_bin                    object
ir_missing                  bool
n_classes                float64
is_binary                   bool
nclasses_bin              object
dataset_size             float64
size_log                 float64
size_bin                  object
size_missing                bool
task3                     object
model_family3             object



,within_paper_group_id,paper_id,doi,article_url,dataset_canonical,strategy,metric_used,y_baseline,y_treatment,delta_y,ir,ir_log,ir_bin,ir_missing,n_classes,is_binary,nclasses_bin,dataset_size,size_log,size_bin,size_missing,task3,model_family3
0,19,arxiv:1710.05381,10.1016/j.neunet.2018.07.011,https://arxiv.org/pdf/1710.05381v2,ImageNet (ILSVRC-2012),oversampling,metric_aucroc,0.93394,0.91870,-0.016318,NaN,NaN,missing,True,1000.0,False,>10,NaN,NaN,missing,True,vision,neural
1,19,arxiv:1710.05381,10.1016/j.neunet.2018.07.011,https://arxiv.org/pdf/1710.05381v2,ImageNet (ILSVRC-2012),undersampling,metric_aucroc,0.93394,0.89600,-0.040624,NaN,NaN,missing,True,1000.0,False,>10,NaN,NaN,missing,True,vision,neural
2,62,arxiv:1807.02608,None,https://arxiv.org/pdf/1807.02608v1,LIDC,oversampling,metric_accuracy,0.66360,0.63312,-0.045931,NaN,NaN,missing,True,5.0,False,3-10,829.0,2.918555,medium,False,tabular,tree_ensemble
3,63,arxiv:1809.02596,None,https://arxiv.org/pdf/1809.02596v1,Credit-card fraud detection,oversampling,metric_f1_binary,0.88800,0.89100,0.003378,NaN,NaN,missing,True,2.0,True,2,NaN,NaN,missing,True,tabular,tree_ensemble
4,64,arxiv:1809.02596,None,https://arxiv.org/pdf/1809.02596v1,Credit-card fraud detection,other,metric_f1_binary,0.12400,0.86300,5.959677,NaN,NaN,missing,True,2.0,True,2,NaN,NaN,missing,True,tabular,neural
5,64,arxiv:1809.02596,None,https://arxiv.org/pdf/1809.02596v1,Credit-card fraud detection,oversampling,metric_f1_binary,0.12400,0.23300,0.879032,NaN,NaN,missing,True,2.0,True,2,NaN,NaN,missing,True,tabular,neural
6,65,arxiv:1811.00972,None,https://arxiv.org/pdf/1811.00972v2,Data-1,cost_sensitive,metric_f1_binary,0.75400,0.98700,0.309019,12.0,1.079181,>=10,False,2.0,True,2,NaN,NaN,missing,True,tabular,neural
7,65,arxiv:1811.00972,None,https://arxiv.org/pdf/1811.00972v2,Data-1,other,metric_f1_binary,0.75400,0.97700,0.295756,12.0,1.079181,>=10,False,2.0,True,2,NaN,NaN,missing,True,tabular,neural
